# CMSC 320 (Checkpoint 2): County-Level Data Cleaning & Aggregation

Combine 2023 BLS, poverty/income, and education data sources into a single row per county/county-equivalent, using FIPS (state-national) identifiers. Running all five code cells writes `county_aggregated_2023.csv` which represents the aggregated dataset we will use for analysis.

IMPORTANT: FIPS should be treated as the unique identifier for counties. All data is for 2023 but education source is filtered for data where lvl is received 2019–2023. All data is county-level but the raw sources contain aggregated-lvl entries which are filtered out.

Interestingly, education does contain counts for four mutually exclusive adult educational groups. I found the sum of the population aged 25 and older for the 2019–2023 period across all educations, which is included as `population_25_plus_2019_2023` but is still not a total pop. That can be found in `population_2023`. I also included the pops for the education groups but not sure how useful it will be.

#### BLS 2023

In [1]:
import pandas as pd

In [2]:
bls = pd.read_excel("data/BLS_County_Labor_Force_2023.xlsx", sheet_name="laucnty23", skiprows=1, dtype=str)
print(f"{bls.shape[0]} rows")
bls = bls[bls["Year"] == "2023"]

# Get full FIPS code
bls["fips"] = bls["State FIPS Code"].str.zfill(2) + bls["County FIPS Code"].str.zfill(3)

# Select what we need with less generic non-whitespace names
new_cols = {
    "County Name/State Abbreviation": "county_name",
    "Labor Force": "labor_force_2023",
    "Employed": "employed_2023",
    "Unemployed": "unemployed_2023",
    "Unemployment Rate (%)": "unemployment_rate_2023"
}
bls = bls[[
    "fips",
    "County Name/State Abbreviation",
    "Labor Force",
    "Employed",
    "Unemployed",
    "Unemployment Rate (%)"
    ]].rename(columns=new_cols)

# Make sure to parse
bls_numbers = ["labor_force_2023", "employed_2023", "unemployed_2023", "unemployment_rate_2023"]
bls[bls_numbers] = bls[bls_numbers].apply(pd.to_numeric)
display(bls.head())

3224 rows


,fips,county_name,labor_force_2023,employed_2023,unemployed_2023,unemployment_rate_2023
0,01001,"Autauga County, AL",27823,27227,596,2.1
1,01003,"Baldwin County, AL",113708,111245,2463,2.2
2,01005,"Barbour County, AL",8703,8362,341,3.9
3,01007,"Bibb County, AL",8617,8409,208,2.4
4,01009,"Blount County, AL",26568,26026,542,2.0


#### Poverty 2023

Note: the CSV is formatted so each county has separate `Attribute`/`Value` rows, so we select the needed attributes and put them into a column-friendly format so that there is one row per FIPS code. Also we have a rucc score which is nice.

In [3]:
poverty = pd.read_csv("data/Poverty2023.csv", dtype={"FIPS_Code": str})  # Keep fips as strs

# Get the needed attrs, fix up data to aggregate the multiple rows/county into one
# Attrs are available w/ a value which tells us the county's poverty_rate for example
poverty_names = {
    "PCTPOVALL_2023": "poverty_rate_2023",
    "PCTPOV017_2023": "child_poverty_rate_2023",
    "MEDHHINC_2023": "median_household_income_2023",
    "Rural_Urban_Continuum_Code_2023": "rural_urban_continuum_code_2023"
}
poverty = poverty[poverty["Attribute"].isin(poverty_names)]
poverty["fips"] = poverty["FIPS_Code"].str.zfill(5)
# ending "000" means aggregate totals, we want only individual county-level records
poverty = poverty[poverty["fips"].str[-3:] != "000"]
poverty = poverty.pivot(index="fips", columns="Attribute", values="Value").reset_index()
poverty = poverty.rename(columns=poverty_names)

print("Education conversion/missing values:", poverty.isna().sum())  # Should be minimal
print("Poverty/income county records:", len(poverty))    # Should be about 3144
print("Duplicate FIPS:", poverty["fips"].duplicated().sum())

Education conversion/missing values: Attribute
fips                               0
median_household_income_2023       1
child_poverty_rate_2023            1
poverty_rate_2023                  1
rural_urban_continuum_code_2023    0
dtype: int64
Poverty/income county records: 3144
Duplicate FIPS: 0


#### Education 2023 and 25+ adult population
Note: I figured we can use the numbers here to find 25+ aged pop which might be nice since it could go hand-in-hand with employment (entering the workforce age)

So `population_25_plus_2019_2023` is the 25+ aged adults in the county that got their education 2019-2023

In [4]:
education = pd.read_csv("data/Education2023.csv", encoding="latin1", dtype={"FIPS Code": str})  # latin1 encoding

population_attributes = [
    "Less than high school graduate, 2019-23",
    "High school graduate (or equivalency), 2019-23",
    "Some college or associate degree, 2019-23",
    "Bachelor's degree or higher, 2019-23"
]
bachelors_attribute = "Percent of adults with a bachelor's degree or higher, 2019-23"

education = education[education["Attribute"].isin(population_attributes + [bachelors_attribute])]
education["fips"] = education["FIPS Code"].str.zfill(5)
education = education[education["fips"].str[-3:] != "000"]   # Keep only county-level (these are state/country-level)

# Fix the multiple row problem again
education = education.pivot(index="fips", columns="Attribute", values="Value").reset_index()
education_columns = population_attributes + [bachelors_attribute] # We want all lvls together
education[education_columns] = education[education_columns].apply(pd.to_numeric)

print("Education conversion/missing values:", education.isna().sum())  # Should be minimal

education["population_25_plus_2019_2023"] = education[population_attributes].sum(axis=1)
education = education[
    ["fips", "population_25_plus_2019_2023", bachelors_attribute] + population_attributes].rename(
    columns={bachelors_attribute: "bachelors_pct_2019_2023"})
print("Education county records:", len(education))   # Ideally close to 3144
print("Duplicate FIPS:", education["fips"].duplicated().sum())

Education conversion/missing values: Attribute
fips                                                             0
Bachelor's degree or higher, 2019-23                             0
High school graduate (or equivalency), 2019-23                   0
Less than high school graduate, 2019-23                          0
Percent of adults with a bachelor's degree or higher, 2019-23    0
Some college or associate degree, 2019-23                        0
dtype: int64
Education county records: 3222
Duplicate FIPS: 0


#### Population 2023
Straightforward, we find the county-lvl population

In [5]:
population = pd.read_csv("data/PopulationEstimates2023.csv", encoding="latin1", dtype={"FIPStxt": str})

# Filter for county-lvl and correct attr
population["fips"] = population["FIPStxt"].str.zfill(5)
population = population[
    (population["Attribute"] == "POP_ESTIMATE_2023") &
    (population["fips"].str[-3:] != "000")
][["fips", "Value"]]

population = population.rename(columns={"Value": "population_2023"})
population["population_2023"] = population["population_2023"].astype(int)

print("Population county records:", len(population)) # Should be close to 3144
print("Duplicate FIPS:", population["fips"].duplicated().sum())

Population county records: 3222
Duplicate FIPS: 0


#### Merging the three dataframes

In [6]:
# Conduct inner joins on the three sources, remove na
bls_poverty = bls.merge(poverty, on="fips", how="inner", validate="one_to_one")
county = bls_poverty.merge(education, on="fips", how="inner", validate="one_to_one")
county = county.merge(population, on="fips", how="inner", validate="one_to_one")

# Should be minimial
print("FIPS lost after poverty merge:", len(set(bls["fips"]) - set(bls_poverty["fips"])))
print("FIPS lost after education merge:", len(set(bls_poverty["fips"]) - set(county["fips"])))

FIPS lost after poverty merge: 78
FIPS lost after education merge: 0


#### Export

In [7]:
# Assert correct types
count_columns = ["population_25_plus_2019_2023", "labor_force_2023", "employed_2023",
                 "unemployed_2023", "median_household_income_2023", "rural_urban_continuum_code_2023"]
county[count_columns] = county[count_columns].astype(int)
percent_columns = ["unemployment_rate_2023", "poverty_rate_2023", "child_poverty_rate_2023", "bachelors_pct_2019_2023"]

print("Duplicate FIPS:", county["fips"].duplicated().sum()) # Should be 0 by now
print("Missing values:\n", county.isna().sum())

# Summary stats of the data
print("Data types:\n", county.dtypes)
print("Summary statistics:\n", county.describe())

county.to_csv("county_aggregated_2023.csv", index=False)
print("Aggregate dataset in county_aggregated_2023.csv")

Duplicate FIPS: 0
Missing values:
 fips                                              0
county_name                                       0
labor_force_2023                                  0
employed_2023                                     0
unemployed_2023                                   0
unemployment_rate_2023                            0
median_household_income_2023                      0
child_poverty_rate_2023                           0
poverty_rate_2023                                 0
rural_urban_continuum_code_2023                   0
population_25_plus_2019_2023                      0
bachelors_pct_2019_2023                           0
Less than high school graduate, 2019-23           0
High school graduate (or equivalency), 2019-23    0
Some college or associate degree, 2019-23         0
Bachelor's degree or higher, 2019-23              0
population_2023                                   0
dtype: int64
Data types:
 fips                                                  s